import sys
sys.path.insert(0, '.')
from polaris_test_utils import *

tr = TestResult()

In [9]:
import sys
sys.path.insert(0, '.')
from polaris_test_utils import *

tr = TestResult()

In [11]:
# T01: OAuth2 token acquisition (watchdog principal, default realm)
print("T01: OAuth2 token acquisition")
try:
    token = get_watchdog_token(WATCHDOG_CLIENT_ID, WATCHDOG_SECRET)
    tr.record("T01_auth", True, "token acquired")
except AssertionError as e:
    tr.record("T01_auth", False, str(e))
    print("⛔ Cannot continue without token")
    raise

T01: OAuth2 token acquisition
  ❌ T01_auth: Watchdog token failed: 401 {"error":"unauthorized_client","error_description":"The client is not authorized","error_uri":null}
⛔ Cannot continue without token


AssertionError: Watchdog token failed: 401 {"error":"unauthorized_client","error_description":"The client is not authorized","error_uri":null}

In [5]:
# T02: Watchdog catalog accessible (via catalog API)
# Note: management /api/management/v1/catalogs requires service_admin privilege.
# watchdog-principal only has catalog-level access — use catalog API instead.
print("\nT02: Watchdog catalog accessible")
r = requests.get(
    f"{BASE_CAT}/{WATCHDOG_CATALOG}/namespaces",
    headers=h(token)
)
passed = r.status_code == 200
detail = f"HTTP {r.status_code}"
if passed:
    ns_list = r.json().get("namespaces", [])
    detail = f"catalog reachable, namespaces: {ns_list}"
tr.record("T02_catalog_accessible", passed, detail)


T02: Watchdog catalog accessible
  ✅ T02_catalog_accessible: catalog reachable, namespaces: []


In [6]:
# T03: Namespace CRUD (within watchdog-catalog)
print("\nT03: Namespace CRUD")

r_create = create_namespace(catalog=WATCHDOG_CATALOG, ns=WATCHDOG_NAMESPACE, token=token)
passed_create = r_create.status_code in (200, 201)
tr.record("T03a_namespace_create", passed_create, f"HTTP {r_create.status_code}")

r_get = requests.get(
    f"{BASE_CAT}/{WATCHDOG_CATALOG}/namespaces/{WATCHDOG_NAMESPACE}",
    headers=h(token)
)
passed_get = r_get.status_code == 200
tr.record("T03b_namespace_read", passed_get, f"HTTP {r_get.status_code}")

tr.record("T03_namespace_crud", passed_create and passed_get,
    "create + read OK" if passed_create and passed_get else "FAILED")


T03: Namespace CRUD
  ❌ T03a_namespace_create: HTTP 500
  ✅ T03b_namespace_read: HTTP 200
  ❌ T03_namespace_crud: FAILED


In [7]:
# Run this debug cell before re-running T03
print("Debug: namespace create response")
r_debug = create_namespace(catalog=WATCHDOG_CATALOG, ns=WATCHDOG_NAMESPACE, token=token)
print(f"Status: {r_debug.status_code}")
print_response(r_debug, "Namespace create")

Debug: namespace create response
Status: 409

Namespace create:
  Status:     409
  Request-Id: 626665e0-9248-4452-921e-0535b85a9a56_0000000000000000094
  Body: {
  "error": {
    "message": "Cannot create namespace watchdog-ns. Namespace already exists",
    "type": "AlreadyExistsException",
    "code": 409
  }
}


In [ ]:
# T04: Table CRUD
print("\nT04: Table CRUD")

r_create = requests.post(
    f"{BASE_CAT}/{WATCHDOG_CATALOG}/namespaces/{WATCHDOG_NAMESPACE}/tables",
    headers=h(token),
    json={
        "name": WATCHDOG_TABLE,
        "location": f"s3a://data-catalog-bucket/{WATCHDOG_CATALOG}/{WATCHDOG_NAMESPACE}/{WATCHDOG_TABLE}/",
        "schema": {
            "type": "struct",
            "fields": [
                {"id": 1, "name": "id",    "type": "long",   "required": True},
                {"id": 2, "name": "value", "type": "string", "required": False}
            ]
        }
    }
)
passed_create = r_create.status_code in (200, 201)
tr.record("T04a_table_create", passed_create, f"HTTP {r_create.status_code}")

r_get = requests.get(
    f"{BASE_CAT}/{WATCHDOG_CATALOG}/namespaces/{WATCHDOG_NAMESPACE}/tables/{WATCHDOG_TABLE}",
    headers=h(token)
)
passed_get = r_get.status_code == 200
tr.record("T04b_table_read", passed_get, f"HTTP {r_get.status_code}")

tr.record("T04_table_crud", passed_create and passed_get,
    "create + read OK" if passed_create and passed_get else "FAILED")

In [ ]:
# T05: RBAC enforcement (negative test)
# Use root token to find a real existing catalog name,
# then try to DELETE it with the watchdog token.
# Polaris must reach the RBAC check (403), not 404.
print("\nT05: RBAC enforcement — watchdog must NOT delete real catalogs")

# Get a real catalog name using root token
r_list = requests.get(f"{BASE_MGMT}/catalogs", headers=h(root_token()))
all_catalogs = [c["name"] for c in r_list.json().get("catalogs", [])]
# Pick any catalog that is NOT the watchdog's own catalog
targets = [c for c in all_catalogs if c != WATCHDOG_CATALOG]

if not targets:
    tr.record("T05_rbac_enforcement", True,
        "skipped — no other catalog exists to test against")
else:
    target = targets[0]
    r = requests.delete(f"{BASE_MGMT}/catalogs/{target}", headers=h(token))
    # Must be 403 — Polaris reached RBAC check and rejected
    # 404 is ambiguous (catalog not found → RBAC never evaluated)
    passed = r.status_code == 403
    tr.record("T05_rbac_enforcement", passed,
        f"HTTP {r.status_code} on DELETE {target} "
        f"({'RBAC enforced ✅' if passed else 'RBAC BROKEN ❌' if r.status_code == 200 else '404=ambiguous, catalog may not exist'})")

In [ ]:
# T06: Health correlation — detect entity_version mismatch
print("\nT06: Health correlation")
r_health = requests.get(f"{POLARIS_URL}/q/health")
health_up = r_health.status_code == 200 and r_health.json().get("status") == "UP"
write_failed = not tr.results.get("T03_namespace_crud", {}).get("passed", True)

if write_failed and health_up:
    tr.record("T06_health_correlation", False,
        "⚠️  entity_version mismatch suspected — health=UP but write failed")
    print()
    print("⚠️  ENTITY VERSION MISMATCH DETECTED")
    print("   Fix (ask system engineer to run):")
    print("     kubectl rollout restart deployment/benchmarks-polaris -n datahub-hynix")
    print("   DO NOT restart PostgreSQL")
else:
    tr.record("T06_health_correlation", True,
        f"health={'UP' if health_up else 'DOWN'} write={'OK' if not write_failed else 'FAIL'}")

In [ ]:
# Cleanup — only watchdog's own table + namespace (not the whole catalog)
print("\nCleanup...")
requests.delete(
    f"{BASE_CAT}/{WATCHDOG_CATALOG}/namespaces/{WATCHDOG_NAMESPACE}/tables/{WATCHDOG_TABLE}",
    headers=h(token)
)
requests.delete(
    f"{BASE_CAT}/{WATCHDOG_CATALOG}/namespaces/{WATCHDOG_NAMESPACE}",
    headers=h(token)
)
print("  table + namespace deleted (catalog + principal kept for next run)")

In [ ]:
# Summary
print()
all_passed = tr.summary()
print()
if all_passed:
    print("✅ Polaris fully operational")
else:
    print("❌ Issues detected — check failed tests above")
    print()
    print("Common fixes:")
    print("  (this notebook has no kubectl access — report below to system engineer)")
    print("  T01 fail: ask to check pod status: kubectl get pods -n datahub-hynix | grep polaris")
    print("  T03 fail + T06 warn: ask to restart: kubectl rollout restart deployment/benchmarks-polaris")
    print("  T05 fail: CHECK RBAC IMMEDIATELY — privileges too broad")